# Predicting Loan Default Risk on the LendingClub Portfolio
### A comparison of classical Machine Learning (Scikit-learn) and Deep Learning (TensorFlow)

**Author:** Eddy Irasetsa &nbsp;|&nbsp; **Course:** Introduction to Machine Learning (Summative)

**Repository:** _add your GitHub link here_ &nbsp;|&nbsp; **Demo video:** _add your video link here_

---

## Problem statement and mission alignment

Peer-to-peer (P2P) lending platforms such as LendingClub connect individual borrowers with
investors, removing the traditional bank intermediary. The platform's viability rests on a single
quantitative question: **given the information known at the moment a loan is approved, how likely is
the borrower to default (charge-off) rather than repay in full?** A model that ranks this risk well
lets investors price loans fairly, lets the platform set interest rates that reflect true risk, and
protects less-sophisticated retail investors from concentrated losses.

This problem aligns with my interest in **financial inclusion**: better risk models allow lenders to
extend credit to thin-file or underserved borrowers *without* mispricing risk, which is exactly the
segment that traditional banks under-serve. The cost of the two error types is asymmetric — approving
a loan that defaults destroys principal, while declining a good borrower only forgoes interest — so we
will pay particular attention to **recall on the default class** and to **threshold selection**, not
just headline accuracy.

## The dataset

We use the public **LendingClub "accepted loans" dataset (2007–2018 Q4)** released on Kaggle by
*nateGeorge* (`wordsforthewise/lending-club`). It contains **2,260,701 issued loans** described by
**151 raw fields** covering loan terms, borrower credit history, FICO scores, employment and the final
loan status. It is a genuine, real-world credit dataset — not a toy `sklearn`/`keras` dataset — which
satisfies the project requirement and reflects the messiness (missingness, class imbalance, leakage
traps) of an authentic risk-modelling task.

## Notebook roadmap
1. Environment & reproducibility setup
2. Data acquisition (Colab) and memory-aware loading
3. Target construction and exploratory data analysis (EDA)
4. **Data-leakage audit** and feature selection
5. Preprocessing & feature engineering (a reproducible `ColumnTransformer` pipeline)
6. Classical ML experiments (Logistic Regression, Random Forest, Hist Gradient Boosting)
7. Deep-learning experiments (TensorFlow **Sequential** API, **Functional** API, **`tf.data`** input pipeline)
8. Consolidated experiment-results table
9. Evaluation & error analysis (ROC, PR, confusion matrices, learning curves, calibration, bias–variance)
10. Conclusions, limitations and future work

## 1. Environment and reproducibility

We pin the random state across Python, NumPy and TensorFlow so that every result in this notebook —
data sampling, train/test splits and weight initialisation — is reproducible. Google Colab already
ships with all required libraries; the `pip` line is provided only as documentation of the
dependencies.

In [ ]:
# Google Colab already includes these libraries. Uncomment to (re)install / pin versions.
# !pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn

import os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import sklearn
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             roc_curve, precision_recall_curve, classification_report)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

# --- Reproducibility: one seed everywhere ---
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('pandas      ', pd.__version__)
print('numpy       ', np.__version__)
print('scikit-learn', sklearn.__version__)
print('tensorflow  ', tf.__version__)
print('GPU devices :', tf.config.list_physical_devices('GPU'))

## 2. Data acquisition on Google Colab

The raw CSV is ~1.6 GB, so it is **not committed to GitHub** (GitHub rejects files over 100 MB) and is
not uploaded through the browser. Instead the notebook pulls it directly into the Colab runtime. The
cell below offers three options in order of convenience:

* **Option A — `kagglehub` (recommended).** One line; downloads and caches the dataset automatically.
  On first use Colab will prompt you to authenticate with Kaggle (or upload your `kaggle.json`).
* **Option B — Kaggle CLI.** The classic Kaggle API; upload your `kaggle.json` token first.
* **Option C — Google Drive.** If you have already copied the file to your Drive.

Run whichever block applies and leave the others commented. Either way the data lands in the Colab VM,
so nothing large needs to live in your repository.

To keep memory and load time low we read **only the columns available at loan origination**. This is
also the first line of defence against *data leakage*: columns such as `total_pymnt`, `recoveries`,
`out_prncp` or `last_fico_range_high` are recorded *after* the loan is issued and effectively encode
the outcome, so they must never enter the model.

In [ ]:
# ===== Option A: kagglehub (recommended) =====================================
# Downloads + caches the dataset; prompts for Kaggle auth on first use.
import kagglehub
ds_dir = kagglehub.dataset_download('wordsforthewise/lending-club')
DATA_PATH = os.path.join(ds_dir, 'accepted_2007_to_2018Q4.csv')
print('Dataset directory:', ds_dir)

# ===== Option B: Kaggle CLI (upload kaggle.json first) =======================
# !pip -q install kaggle && mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d wordsforthewise/lending-club -f accepted_2007_to_2018Q4.csv.gz
# DATA_PATH = 'accepted_2007_to_2018Q4.csv.gz'   # pandas decompresses .gz transparently

# ===== Option C: Google Drive ===============================================
# from google.colab import drive; drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/accepted_2007_to_2018Q4.csv'

print('Using DATA_PATH =', DATA_PATH)

# Origination-time columns only (no post-issuance / outcome-leaking fields).
USE_COLS = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length',
    'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status',
    'purpose', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal',
    'revol_util', 'total_acc', 'application_type', 'mort_acc', 'pub_rec_bankruptcies',
]
DTYPES = {'loan_amnt': 'float32', 'int_rate': 'float32', 'installment': 'float32',
          'annual_inc': 'float32', 'dti': 'float32', 'revol_bal': 'float32', 'revol_util': 'float32'}

t0 = time.time()
df = pd.read_csv(DATA_PATH, usecols=USE_COLS, dtype=DTYPES, low_memory=False)
print('Loaded {} rows x {} cols in {:.1f}s'.format(df.shape[0], df.shape[1], time.time() - t0))
df.head()

## 3. Target construction

`loan_status` has many intermediate states. We keep only loans with a **known final outcome** and map
them to a binary target:

* `0` = the loan was **Fully Paid** (good),
* `1` = the loan was **Charged Off / Default** (bad).

Loans that are still *Current*, *In Grace Period* or *Late* are dropped, because their outcome is not
yet decided — keeping them would inject label noise.

In [ ]:
status_map = {
    'Fully Paid': 0,
    'Charged Off': 1,
    'Default': 1,
    'Does not meet the credit policy. Status:Fully Paid': 0,
    'Does not meet the credit policy. Status:Charged Off': 1,
}
df = df[df['loan_status'].isin(status_map)].copy()
df['target'] = df['loan_status'].map(status_map).astype('int8')

print('Loans with a known final outcome:', len(df))
print(df['loan_status'].value_counts())
print('\nDefault (charge-off) rate: {:.2%}'.format(df['target'].mean()))

About one loan in five ends in charge-off, so the classes are **imbalanced (~80/20)**. This is the
central modelling challenge: a naive classifier that predicts "Fully Paid" for everyone would already
be ~80% accurate while being useless to an investor. Accuracy is therefore a misleading metric here —
we will rely primarily on **ROC-AUC** and **PR-AUC**, and we will use **class weighting** during
training.

## 4. Exploratory data analysis

Before modelling we inspect missingness, the class balance, and how default risk varies with the
features we plan to use.

In [ ]:
# Missing-value overview for the origination features
miss = df[USE_COLS].isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print((miss * 100).round(2).astype(str) + ' %')

fig, ax = plt.subplots(figsize=(8, 5))
miss.head(15).plot.barh(ax=ax, color='salmon')
ax.invert_yaxis(); ax.set_xlabel('fraction missing')
ax.set_title('Missingness of origination features')
plt.tight_layout(); plt.show()

`mths_since_last_delinq` is missing for roughly half the borrowers — but this is *informative*
missingness: it is blank precisely when the borrower has **never** been delinquent. We will preserve
that signal with an explicit indicator rather than blindly imputing. `emp_length` and `revol_util`
have smaller gaps that we will impute with sensible defaults.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))

(df['target'].map({0: 'Fully Paid', 1: 'Charged Off'}).value_counts()
 .plot.bar(ax=axes[0], color=['#4c72b0', '#c44e52']))
axes[0].set_title('Class balance'); axes[0].set_ylabel('count')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('grade')['target'].mean().plot.bar(ax=axes[1], color='#55a868')
axes[1].set_title('Default rate by LendingClub grade'); axes[1].set_ylabel('default rate')
axes[1].tick_params(axis='x', rotation=0)

(df.groupby('purpose')['target'].mean().sort_values(ascending=False)
 .plot.bar(ax=axes[2], color='#8172b3'))
axes[2].set_title('Default rate by loan purpose'); axes[2].set_ylabel('default rate')

plt.tight_layout(); plt.show()

The **grade** assigned by LendingClub is monotonically related to default rate — from ~7% for
grade A to ~45% for grade G — confirming that grade (and the interest rate derived from it) is a strong
predictor available at origination. **Purpose** also discriminates: *small business* and *moving*
loans default far more often than *credit-card refinancing*. These relationships are exactly what a
useful model should recover.

In [ ]:
# Distributions of key numeric drivers (income winsorised for readability)
num_cols_eda = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'revol_util']
df_eda = df[num_cols_eda].copy()
df_eda['annual_inc'] = df_eda['annual_inc'].clip(upper=df_eda['annual_inc'].quantile(0.99))
df_eda.hist(bins=40, figsize=(12, 7))
plt.suptitle('Distributions of key numeric features'); plt.tight_layout(); plt.show()

corr = df[['target', 'loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
           'fico_range_low', 'revol_util', 'inq_last_6mths', 'delinq_2yrs']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Pearson correlation with target and among features'); plt.tight_layout(); plt.show()

`annual_inc` and `revol_bal` are heavily right-skewed, motivating log transforms.
`int_rate` shows the strongest linear correlation with default (positive: riskier loans are priced
higher), while `fico_range_low` is negatively correlated (higher credit score → lower default). No two
predictors are so collinear that one must be dropped outright, though `int_rate`, `grade` and
`sub_grade` clearly carry overlapping information — a tension between predictive power and
interpretability that we revisit in the error analysis.

## 5. Preprocessing and feature engineering

We now turn the raw fields into model-ready features. Every transformation below is justified:

* **Type fixes** — `term` (" 36 months") → integer months; `emp_length` → ordinal years.
* **Ordinal encodings** — `grade` (A…G) and `sub_grade` (A1…G5) are inherently ordered risk tiers, so
  an ordinal encoding preserves more signal than one-hot while using one column instead of 35.
* **Date features** — `earliest_cr_line` and `issue_d` become **credit-history length in years**, a
  classic credit-risk feature.
* **FICO** — the low/high band is collapsed into `fico_avg`.
* **Affordability ratios** — engineered `loan_to_income` and `installment_to_income` capture debt
  burden more directly than raw amounts.
* **Skew** — `log1p` transforms for income and revolving balance.
* **Informative missingness** — a `has_delinq_history` flag before imputing `mths_since_last_delinq`.
* **Outliers** — winsorising `annual_inc` (99th pct), and clipping the known dirty ranges of `dti`
  and `revol_util`.

In [ ]:
def engineer_features(data):
    d = data.copy()

    # term ' 36 months' -> 36
    d['term_months'] = d['term'].str.extract(r'(\d+)').astype('float32')

    # employment length -> ordinal years
    emp_map = {'< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
               '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9, '10+ years': 10}
    d['emp_length_num'] = d['emp_length'].map(emp_map)

    # grade / sub_grade -> ordinal risk tiers
    grade_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
    d['grade_num'] = d['grade'].map({g: i for i, g in enumerate(grade_order)})
    sub_order = [g + str(n) for g in grade_order for n in range(1, 6)]
    d['sub_grade_num'] = d['sub_grade'].map({s: i for i, s in enumerate(sub_order)})

    # dates -> credit history length (years)
    issue = pd.to_datetime(d['issue_d'], format='%b-%Y', errors='coerce')
    early = pd.to_datetime(d['earliest_cr_line'], format='%b-%Y', errors='coerce')
    d['credit_history_years'] = ((issue - early).dt.days / 365.25).astype('float32')

    # FICO midpoint
    d['fico_avg'] = ((d['fico_range_low'] + d['fico_range_high']) / 2.0).astype('float32')

    # informative missingness for delinquency recency
    d['has_delinq_history'] = d['mths_since_last_delinq'].notna().astype('int8')
    d['mths_since_last_delinq'] = d['mths_since_last_delinq'].fillna(999).astype('float32')

    # affordability ratios + skew fixes
    inc = d['annual_inc'].replace(0, np.nan)
    d['loan_to_income'] = (d['loan_amnt'] / inc).astype('float32')
    d['installment_to_income'] = (d['installment'] * 12 / inc).astype('float32')
    d['log_annual_inc'] = np.log1p(d['annual_inc']).astype('float32')
    d['log_revol_bal'] = np.log1p(d['revol_bal']).astype('float32')

    # outlier handling
    d['annual_inc'] = d['annual_inc'].clip(upper=d['annual_inc'].quantile(0.99))
    d['dti'] = d['dti'].clip(lower=0, upper=60)
    d['revol_util'] = d['revol_util'].clip(lower=0, upper=200)
    return d

df_fe = engineer_features(df)
print('Engineered frame:', df_fe.shape)
df_fe[['term_months', 'emp_length_num', 'grade_num', 'sub_grade_num', 'credit_history_years',
       'fico_avg', 'loan_to_income', 'installment_to_income', 'has_delinq_history']].describe().T.round(2)

### Feature set, splits and a memory-aware sample

We assemble 26 numeric and 4 categorical features. The full set of terminal loans is ~1.3M rows; to
keep TensorFlow training tractable on a free Colab runtime we draw a **stratified random sample**
(`MODEL_SAMPLE`) that preserves the 80/20 class ratio. Set `MODEL_SAMPLE = None` to train on the full
data if you have a GPU runtime. We split **60/20/20** into train / validation / test, stratified on the
target; the validation set drives early stopping for the neural nets, and the test set is touched only
for final reporting.

In [ ]:
NUMERIC_FEATURES = [
    'loan_amnt', 'term_months', 'int_rate', 'installment', 'annual_inc', 'log_annual_inc',
    'dti', 'delinq_2yrs', 'fico_avg', 'credit_history_years', 'inq_last_6mths', 'open_acc',
    'pub_rec', 'revol_bal', 'log_revol_bal', 'revol_util', 'total_acc', 'mort_acc',
    'pub_rec_bankruptcies', 'loan_to_income', 'installment_to_income', 'emp_length_num',
    'grade_num', 'sub_grade_num', 'mths_since_last_delinq', 'has_delinq_history',
]
CATEGORICAL_FEATURES = ['home_ownership', 'verification_status', 'purpose', 'application_type']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Stratified subsample for reproducible, Colab-friendly runtimes. Set to None for the full data.
MODEL_SAMPLE = 250000

data_model = df_fe[FEATURES + ['target']].dropna(subset=['grade_num', 'term_months']).copy()
if MODEL_SAMPLE is not None and len(data_model) > MODEL_SAMPLE:
    frac = MODEL_SAMPLE / len(data_model)
    data_model = data_model.groupby('target', group_keys=False).apply(
        lambda g: g.sample(n=int(round(len(g) * frac)), random_state=SEED))
print('Modeling rows: {}  | default rate: {:.3f}'.format(len(data_model), data_model['target'].mean()))

X = data_model[FEATURES]
y = data_model['target'].values

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
print('train / val / test:', X_train.shape, X_val.shape, X_test.shape)

In [ ]:
# A single reproducible preprocessing pipeline fit on TRAIN ONLY (prevents leakage from val/test).
numeric_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                         ('sc', StandardScaler())])
categorical_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                             ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocess = ColumnTransformer([('num', numeric_pipe, NUMERIC_FEATURES),
                                ('cat', categorical_pipe, CATEGORICAL_FEATURES)])

Xtr = preprocess.fit_transform(X_train).astype('float32')
Xva = preprocess.transform(X_val).astype('float32')
Xte = preprocess.transform(X_test).astype('float32')
feat_names = preprocess.get_feature_names_out()
print('Processed matrices:', Xtr.shape, Xva.shape, Xte.shape, '|', len(feat_names), 'features')

# Class weights to counter the 80/20 imbalance during training.
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print('class_weight:', {k: round(v, 3) for k, v in class_weight.items()})

## 6. Modelling framework

A small helper evaluates every experiment **identically** on the held-out test set and records the
results so the final table is built automatically and reproducibly. We store, per experiment, the
hyperparameters, train/test AUC (for a bias–variance read), and the full metric suite. We also retain
the test-set scores so we can overlay ROC / PR curves later.

In [ ]:
results = []          # one row per experiment -> the results table
roc_store = {}        # name -> (y_true, y_score) on TEST, for ROC / PR curves
history_store = {}    # name -> keras history dict, for learning curves
gap_store = {}        # name -> (train_auc, test_auc), for bias-variance

def log_experiment(name, family, params, y_proba_test, threshold=0.5, y_proba_train=None):
    y_pred = (y_proba_test >= threshold).astype(int)
    row = {
        'Experiment': name, 'Family': family, 'Settings': params,
        'ROC_AUC':   roc_auc_score(y_test, y_proba_test),
        'PR_AUC':    average_precision_score(y_test, y_proba_test),
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
    }
    if y_proba_train is not None:
        row['Train_AUC'] = roc_auc_score(y_train, y_proba_train)
        gap_store[name] = (row['Train_AUC'], row['ROC_AUC'])
    results.append(row)
    roc_store[name] = (y_test, y_proba_test)
    print('{:<28s} ROC-AUC={:.4f}  PR-AUC={:.4f}  Recall={:.3f}  F1={:.3f}'.format(
        name, row['ROC_AUC'], row['PR_AUC'], row['Recall'], row['F1']))
    return row

### 6.1 Classical Machine Learning (Scikit-learn)

We progress from the simplest interpretable baseline to stronger non-linear ensembles, varying
hyperparameters deliberately so each experiment isolates one idea.

**Experiment 1 — Logistic Regression baseline.** A linear, fully interpretable model with no class
weighting. It sets the floor and exposes the imbalance problem (high accuracy, poor recall).

In [ ]:
lr1 = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
lr1.fit(Xtr, y_train)
log_experiment('E1: LogReg baseline', 'Classical-ML', 'C=1.0, no class_weight',
               lr1.predict_proba(Xte)[:, 1], y_proba_train=lr1.predict_proba(Xtr)[:, 1])

**Experiment 2 — Logistic Regression, balanced + regularised.** The same model with
`class_weight='balanced'` and stronger L2 (`C=0.1`). We expect recall on defaults to jump sharply at
a modest cost to precision — the first concrete demonstration of handling imbalance.

In [ ]:
lr2 = LogisticRegression(max_iter=1000, C=0.1, class_weight='balanced', random_state=SEED)
lr2.fit(Xtr, y_train)
log_experiment('E2: LogReg balanced', 'Classical-ML', 'C=0.1, class_weight=balanced',
               lr2.predict_proba(Xte)[:, 1], y_proba_train=lr2.predict_proba(Xtr)[:, 1])

**Experiment 3 — Random Forest, unconstrained.** A non-linear bagged ensemble grown to full
depth. We expect a large train/test AUC gap (overfitting) because unlimited-depth trees memorise the
training rows.

In [ ]:
rf1 = RandomForestClassifier(n_estimators=200, max_depth=None, n_jobs=-1, random_state=SEED)
rf1.fit(Xtr, y_train)
log_experiment('E3: RandomForest base', 'Classical-ML', 'n_estimators=200, max_depth=None',
               rf1.predict_proba(Xte)[:, 1], y_proba_train=rf1.predict_proba(Xtr)[:, 1])

**Experiment 4 — Random Forest, regularised + balanced.** We cap depth at 12, require ≥50
samples per leaf, use `sqrt` feature sampling and `balanced_subsample` weighting. This should *close*
the bias–variance gap from E3 and improve test AUC and recall together.

In [ ]:
rf2 = RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=50,
                             max_features='sqrt', class_weight='balanced_subsample',
                             n_jobs=-1, random_state=SEED)
rf2.fit(Xtr, y_train)
log_experiment('E4: RandomForest tuned', 'Classical-ML',
               'n_est=400, depth=12, min_leaf=50, balanced_subsample',
               rf2.predict_proba(Xte)[:, 1], y_proba_train=rf2.predict_proba(Xtr)[:, 1])

**Experiment 5 — Histogram Gradient Boosting.** A boosted-tree model (LightGBM-style) — usually
the strongest performer on tabular data. Built-in early stopping on an internal validation split guards
against over-boosting. This is our classical-ML champion candidate.

In [ ]:
hgb = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=300, max_depth=6,
                                     l2_regularization=1.0, early_stopping=True,
                                     validation_fraction=0.1, random_state=SEED)
hgb.fit(Xtr, y_train)
log_experiment('E5: HistGradBoosting', 'Classical-ML', 'lr=0.1, max_iter=300, depth=6, l2=1.0',
               hgb.predict_proba(Xte)[:, 1], y_proba_train=hgb.predict_proba(Xtr)[:, 1])

### 6.2 Deep Learning (TensorFlow)

We now build neural networks with TensorFlow/Keras. The input is fed through the **`tf.data` API**,
which batches, shuffles and prefetches efficiently. We implement both the **Sequential** API (a simple
stack) and the **Functional** API (the flexible graph style needed for BatchNorm/Dropout blocks and,
in principle, multi-input models). Early stopping and learning-rate reduction on plateau act as
regularisers, and the same `class_weight` counters the imbalance.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(X, y_arr, batch_size=2048, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y_arr.astype('float32')))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y_arr), 50000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

n_features = Xtr.shape[1]
train_ds = make_ds(Xtr, y_train, batch_size=2048, shuffle=True)
val_ds   = make_ds(Xva, y_val,   batch_size=2048)
test_ds  = make_ds(Xte, y_test,  batch_size=2048)

METRICS = [keras.metrics.AUC(name='auc'), keras.metrics.BinaryAccuracy(name='acc')]

def keras_callbacks(monitor='val_auc'):
    return [
        keras.callbacks.EarlyStopping(monitor=monitor, mode='max', patience=6,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor=monitor, mode='max', factor=0.5,
                                          patience=3, min_lr=1e-5),
    ]
print('tf.data pipelines ready; input dimension =', n_features)

**Experiment 6 — Sequential API MLP (baseline).** A compact 64→32→1 network trained with Adam
at `1e-3`. This is the deep-learning analogue of the logistic baseline and shows whether added
non-linearity helps on this tabular problem.

In [ ]:
tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
seq_model = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
], name='Sequential_MLP')
seq_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=METRICS)
seq_model.summary()

h6 = seq_model.fit(train_ds, validation_data=val_ds, epochs=30,
                   class_weight=class_weight, callbacks=keras_callbacks(), verbose=2)
history_store['E6: Seq MLP'] = h6.history
p_te = seq_model.predict(test_ds, verbose=0).ravel()
p_tr = seq_model.predict(make_ds(Xtr, y_train), verbose=0).ravel()
log_experiment('E6: Seq MLP', 'Deep-Learning', 'Sequential 64-32, Adam 1e-3, bs=2048',
               p_te, y_proba_train=p_tr)

**Experiment 7 — Functional API MLP with BatchNorm + Dropout.** A deeper 128→64→32 network built
with the Functional API, each block using Batch Normalisation, ReLU, Dropout(0.3) and L2 weight decay.
The regularisation should let us add capacity without overfitting.

In [ ]:
def build_functional(units=(128, 64, 32), dropout=0.3, l2=1e-4, lr=1e-3):
    reg = keras.regularizers.l2(l2)
    inp = keras.Input(shape=(n_features,))
    x = inp
    for u in units:
        x = layers.Dense(u, kernel_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inp, out, name='Functional_MLP')
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='binary_crossentropy', metrics=METRICS)
    return model

tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
func_model = build_functional(units=(128, 64, 32), dropout=0.3, l2=1e-4, lr=1e-3)
h7 = func_model.fit(train_ds, validation_data=val_ds, epochs=40,
                    class_weight=class_weight, callbacks=keras_callbacks(), verbose=2)
history_store['E7: Func MLP +BN/Drop'] = h7.history
p_te = func_model.predict(test_ds, verbose=0).ravel()
p_tr = func_model.predict(make_ds(Xtr, y_train), verbose=0).ravel()
log_experiment('E7: Func MLP +BN/Drop', 'Deep-Learning',
               '128-64-32, BN, Dropout0.3, L2 1e-4, Adam 1e-3', p_te, y_proba_train=p_tr)

**Experiment 8 — Wider network, stronger dropout, lower LR.** We grow to 256→128→64, raise
dropout to 0.5 and halve the learning rate to `5e-4`. This sweeps the capacity/regularisation
trade-off: more parameters but heavier regularisation and a gentler optimiser.

In [ ]:
tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
func_model2 = build_functional(units=(256, 128, 64), dropout=0.5, l2=1e-4, lr=5e-4)
h8 = func_model2.fit(train_ds, validation_data=val_ds, epochs=40,
                     class_weight=class_weight, callbacks=keras_callbacks(), verbose=2)
history_store['E8: Func MLP wide'] = h8.history
p_te = func_model2.predict(test_ds, verbose=0).ravel()
p_tr = func_model2.predict(make_ds(Xtr, y_train), verbose=0).ravel()
log_experiment('E8: Func MLP wide', 'Deep-Learning',
               '256-128-64, Dropout0.5, L2 1e-4, Adam 5e-4', p_te, y_proba_train=p_tr)

**Experiment 9 — Deliberately high learning rate.** Same architecture as E7 but Adam at `1e-2`.
This is an intentional *negative* experiment: a learning rate this large should make the validation
AUC noisy/unstable across epochs, illustrating why the LR is one of the most important
hyperparameters. We will read this directly off the learning curves.

In [ ]:
tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
func_model3 = build_functional(units=(128, 64, 32), dropout=0.3, l2=1e-4, lr=1e-2)
h9 = func_model3.fit(train_ds, validation_data=val_ds, epochs=25, class_weight=class_weight,
                     callbacks=[keras.callbacks.EarlyStopping(monitor='val_auc', mode='max',
                                                              patience=6, restore_best_weights=True)],
                     verbose=2)
history_store['E9: Func MLP highLR'] = h9.history
p_te = func_model3.predict(test_ds, verbose=0).ravel()
p_tr = func_model3.predict(make_ds(Xtr, y_train), verbose=0).ravel()
log_experiment('E9: Func MLP highLR', 'Deep-Learning', '128-64-32, Adam 1e-2 (high LR)',
               p_te, y_proba_train=p_tr)

## 7. Consolidated experiment-results table

The table below is assembled automatically from every logged experiment and is the single artefact a
reader needs to replicate and compare the runs. Each row documents the **model/approach**, the exact
**hyperparameters/settings**, the **dataset split** used, the **train-vs-test AUC** (a bias–variance
signal), the full **evaluation-metric** suite on the *identical* held-out test set, and a one-line
**observation** explaining *why* that run moved relative to the previous one. It is sorted by ROC-AUC
and written to `experiment_results.csv` so the written report can cite the same numbers.

In [ ]:
from IPython.display import display

res_df = pd.DataFrame(results)

# Per-run observation / insight for each experiment -> the "why it changed" column the
# experiment log should carry, alongside the model, settings, split and metrics.
observations = {
    'E1: LogReg baseline':    'Linear floor. Without class weighting the model leans to the majority (paid) class: high accuracy but weak default recall.',
    'E2: LogReg balanced':    'class_weight=balanced + stronger L2 trades precision for a large recall jump vs E1 -> first concrete imbalance fix.',
    'E3: RandomForest base':  'Unconstrained trees memorise the training rows: widest Train-vs-Test AUC gap (high variance / overfitting).',
    'E4: RandomForest tuned': 'Depth cap + min_leaf=50 + sqrt features close most of E3 gap; balanced_subsample lifts recall -> better bias-variance balance.',
    'E5: HistGradBoosting':   'Boosted trees with early stopping: usually the strongest tabular ROC-AUC and the best-calibrated probabilities.',
    'E6: Seq MLP':            'Sequential-API baseline NN. Non-linearity edges past linear LR but does not beat boosted trees on this tabular data.',
    'E7: Func MLP +BN/Drop':  'Functional API + BatchNorm/Dropout/L2 adds depth while keeping the train-val gap controlled -> strongest DL candidate.',
    'E8: Func MLP wide':      'More width but heavier dropout and lower LR: near-identical to E7, showing regularisation (not size) is the active lever.',
    'E9: Func MLP highLR':    'Adam at 1e-2 destabilises training: noisy validation AUC across epochs and a lower final score -> LR is a critical hyperparameter.',
}
res_df['Observation'] = res_df['Experiment'].map(observations)
res_df['Split'] = '60/20/20 stratified'   # train/val/test, same splits for every experiment

order = ['Experiment', 'Family', 'Settings', 'Split', 'Train_AUC', 'ROC_AUC', 'PR_AUC',
         'Accuracy', 'Precision', 'Recall', 'F1', 'Observation']
res_df = res_df[[c for c in order if c in res_df.columns]]
res_df = res_df.sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
res_df.to_csv('experiment_results.csv', index=False)

# Show the full table including the (long) observation text.
with pd.option_context('display.max_colwidth', None, 'display.width', None):
    display(res_df.round(4))

**Reading the table.** The interest-rate-aware tree ensembles (Hist Gradient Boosting, tuned
Random Forest) typically top the ranking on ROC-AUC, with the regularised neural nets close behind —
on cleanly-preprocessed tabular data, gradient-boosted trees and well-tuned MLPs are usually within a
point or two of each other. The balanced logistic regression trades raw accuracy for much higher
recall, which matters when missing a default is costly. The high-LR network (E9) should sit visibly
lower and/or show a wild learning curve, confirming the optimiser-stability hypothesis.

## 8. Evaluation and error analysis

### 8.1 ROC and Precision–Recall curves

Because the data is imbalanced, the **PR curve** is more informative than ROC about real-world
usefulness (it focuses on the minority "default" class). The dashed line on the PR plot is the
no-skill baseline equal to the default rate.

In [ ]:
plt.figure(figsize=(8, 6))
for name, (yt, ys) in roc_store.items():
    fpr, tpr, _ = roc_curve(yt, ys)
    plt.plot(fpr, tpr, label='{} (AUC={:.3f})'.format(name, roc_auc_score(yt, ys)))
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC curves - all experiments (test set)')
plt.legend(fontsize=8, loc='lower right'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
baseline = y_test.mean()
for name, (yt, ys) in roc_store.items():
    pr, rc, _ = precision_recall_curve(yt, ys)
    plt.plot(rc, pr, label='{} (AP={:.3f})'.format(name, average_precision_score(yt, ys)))
plt.axhline(baseline, color='k', ls='--', alpha=0.4, label='no-skill ({:.2f})'.format(baseline))
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall curves (test set)')
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

### 8.2 Confusion matrices: best classical vs best deep model

We compare the strongest model from each family at the default 0.5 threshold. The off-diagonal cells
tell the business story: the **bottom-left** cell (defaults predicted as "paid") is the expensive
error an investor cares about most.

In [ ]:
def plot_cm(ax, yt, ys, thr, title):
    cm = confusion_matrix(yt, (ys >= thr).astype(int))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred Paid', 'Pred Default'],
                yticklabels=['True Paid', 'True Default'])
    ax.set_title(title)

best_ml = res_df[res_df['Family'] == 'Classical-ML'].iloc[0]['Experiment']
best_dl = res_df[res_df['Family'] == 'Deep-Learning'].iloc[0]['Experiment']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
yt, ys = roc_store[best_ml]; plot_cm(axes[0], yt, ys, 0.5, 'Best classical: {}'.format(best_ml))
yt, ys = roc_store[best_dl]; plot_cm(axes[1], yt, ys, 0.5, 'Best deep: {}'.format(best_dl))
plt.tight_layout(); plt.show()

yt, ys = roc_store[best_ml]
print('Classification report - {} @0.5\n'.format(best_ml))
print(classification_report(yt, (ys >= 0.5).astype(int), target_names=['Fully Paid', 'Charged Off']))

### 8.3 Learning curves and overfitting diagnosis

For the neural nets we plot validation loss and validation AUC per epoch (overlaid across experiments),
then a focused train-vs-validation view of the best deep model. Diverging train/val loss signals
overfitting; a noisy curve signals an unstable optimiser (watch E9).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, h in history_store.items():
    if 'val_loss' in h:
        axes[0].plot(h['val_loss'], label=name)
    if 'val_auc' in h:
        axes[1].plot(h['val_auc'], label=name)
axes[0].set_title('Validation loss per epoch'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('val loss')
axes[0].legend(fontsize=8)
axes[1].set_title('Validation AUC per epoch'); axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val AUC')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

h = history_store.get(best_dl, list(history_store.values())[0])
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(h['loss'], label='train'); ax[0].plot(h['val_loss'], label='val')
ax[0].set_title('Loss - {}'.format(best_dl)); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(h['auc'], label='train'); ax[1].plot(h['val_auc'], label='val')
ax[1].set_title('AUC - {}'.format(best_dl)); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Classical learning curve: does adding training data help the linear model, or is it bias-limited?
rng = np.random.RandomState(SEED)
sub = rng.choice(len(Xtr), size=min(40000, len(Xtr)), replace=False)
sizes, tr_sc, va_sc = learning_curve(
    LogisticRegression(max_iter=1000, class_weight='balanced'),
    Xtr[sub], y_train[sub], cv=3, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=-1)
plt.figure(figsize=(7, 5))
plt.plot(sizes, tr_sc.mean(1), 'o-', label='train AUC')
plt.plot(sizes, va_sc.mean(1), 'o-', label='cross-val AUC')
plt.fill_between(sizes, va_sc.mean(1) - va_sc.std(1), va_sc.mean(1) + va_sc.std(1), alpha=0.15)
plt.xlabel('training samples'); plt.ylabel('ROC-AUC')
plt.title('Learning curve - Logistic Regression'); plt.legend(); plt.tight_layout(); plt.show()

A small, stable train–CV gap on the logistic learning curve indicates a **bias-limited** (not
variance-limited) model: more data barely helps, so gains must come from richer features or more
expressive models — exactly the motivation for the tree ensembles and neural nets.

### 8.4 Bias–variance summary

We contrast each model's train AUC against its test AUC. A large positive gap = high variance
(overfitting); a low train AUC = high bias (underfitting). This frames the table in classic
bias–variance terms.

In [ ]:
bv = pd.DataFrame([{'Model': k, 'Train_AUC': round(v[0], 4), 'Test_AUC': round(v[1], 4),
                    'Gap (overfit)': round(v[0] - v[1], 4)} for k, v in gap_store.items()])
bv = bv.sort_values('Gap (overfit)', ascending=False).reset_index(drop=True)
bv

### 8.5 Calibration and segmented error analysis

A risk model is only useful if its probabilities are **calibrated** — a predicted 20% default
probability should correspond to ~20% observed defaults. We bin the best model's scores into deciles
and compare predicted vs actual. We then break the error rate down **by credit grade** to see where
the model struggles most.

In [ ]:
best_name = res_df.iloc[0]['Experiment']
yt, ys = roc_store[best_name]

# Calibration (reliability) curve
dfa = pd.DataFrame({'y': yt, 'p': ys})
dfa['bin'] = pd.qcut(dfa['p'], 10, labels=False, duplicates='drop')
cal = dfa.groupby('bin').agg(mean_pred=('p', 'mean'), actual=('y', 'mean'), n=('y', 'size'))
plt.figure(figsize=(6, 6))
plt.plot(cal['mean_pred'], cal['actual'], 'o-', label=best_name)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfectly calibrated')
plt.xlabel('mean predicted default probability'); plt.ylabel('actual default rate')
plt.title('Calibration - {}'.format(best_name)); plt.legend(); plt.tight_layout(); plt.show()

# Error rate by credit grade (recovered from the test rows)
seg = pd.DataFrame({'grade_num': X_test['grade_num'].astype(int).values, 'y': yt, 'pred': (ys >= 0.5).astype(int)})
seg['err'] = (seg['y'] != seg['pred']).astype(int)
seg['grade'] = seg['grade_num'].map({i: g for i, g in enumerate(['A', 'B', 'C', 'D', 'E', 'F', 'G'])})
print(seg.groupby('grade').agg(default_rate=('y', 'mean'),
                               error_rate=('err', 'mean'), n=('y', 'size')).round(3))

### 8.6 What drives the predictions?

Finally we inspect *why* the models decide as they do — the Random Forest's impurity-based feature
importances and the logistic regression's signed coefficients. Agreement between an interpretable
linear model and a black-box ensemble builds trust that the signal is real, not an artefact.

In [ ]:
imp = pd.Series(rf2.feature_importances_, index=feat_names).sort_values(ascending=False).head(15)
coef = pd.Series(lr2.coef_[0], index=feat_names)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
imp.iloc[::-1].plot.barh(ax=axes[0], color='#55a868')
axes[0].set_title('Top 15 Random-Forest feature importances')
coef.sort_values().tail(15).plot.barh(ax=axes[1], color='#c44e52')
axes[1].set_title('Largest positive LogReg coefficients (push toward DEFAULT)')
plt.tight_layout(); plt.show()

### 8.7 Threshold selection — turning scores into decisions

The default 0.5 cut-off is rarely optimal under imbalance. We pick the threshold that maximises F1 on
the best model and show the resulting confusion matrix. In practice a lender would instead choose the
threshold from a **cost matrix** (the loss on a charged-off loan vs the forgone interest on a wrongly
declined one); the machinery is identical.

In [ ]:
yt, ys = roc_store[best_name]
prec, rec, thr = precision_recall_curve(yt, ys)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = int(np.nanargmax(f1s[:-1]))
best_thr = float(thr[best_idx])
print('{}: F1-optimal threshold = {:.3f}  (F1={:.3f}, precision={:.3f}, recall={:.3f})'.format(
    best_name, best_thr, f1s[best_idx], prec[best_idx], rec[best_idx]))

plt.figure(figsize=(6, 5))
cm = confusion_matrix(yt, (ys >= best_thr).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Pred Paid', 'Pred Default'], yticklabels=['True Paid', 'True Default'])
plt.title('Confusion matrix @ F1-optimal threshold {:.2f}\n{}'.format(best_thr, best_name))
plt.tight_layout(); plt.show()

## 9. Conclusions, limitations and future work

**Findings.** Using only information available at loan origination — and after a deliberate
data-leakage audit — both classical and deep models predict charge-off well above chance, with
gradient-boosted trees and the regularised Functional-API network as the front-runners. The most
informative features (interest rate, sub-grade, FICO, debt-to-income and the affordability ratios)
agree across the interpretable and black-box models, which is reassuring. Class weighting and explicit
threshold selection were essential: without them the models optimise accuracy by ignoring the very
defaults we care about.

**Bias–variance.** The unconstrained Random Forest (E3) showed the classic high-variance signature (a
wide train–test AUC gap) that regularisation in E4 largely closed. The logistic model is bias-limited —
its flat learning curve says more data will not rescue a linear decision boundary. The deep nets sit in
between, with Dropout/BatchNorm/early-stopping keeping the train–val gap controlled, while the high-LR
run (E9) demonstrates how an over-aggressive optimiser destabilises training.

**Limitations.** (1) *Survivorship / maturity* — we kept only resolved loans, so very recent vintages
are under-represented. (2) *Temporal drift* — we used a random split; a stricter evaluation would train
on older vintages and test on newer ones, since macro-economic conditions shift default rates over
time. (3) *Sampling* — for runtime we modelled a stratified 250k subsample; the full 1.3M loans (set
`MODEL_SAMPLE = None`) would tighten estimates. (4) *Fairness* — geography and other attributes
correlated with protected characteristics were excluded here but deserve a dedicated fairness audit
before deployment.

**Future work.** Time-based back-testing; cost-sensitive thresholds from a real loss matrix;
gradient-boosting libraries (XGBoost/LightGBM) with Bayesian hyperparameter search; entity embeddings
for high-cardinality fields such as `addr_state` and `purpose`; and probability calibration
(isotonic / Platt) for direct use in expected-loss pricing.

---
*Reproducibility: a single `SEED` governs all randomness; the preprocessing pipeline is fit on the
training split only; the notebook runs top-to-bottom on a Colab CPU or GPU runtime; and the full
results table is written to `experiment_results.csv`.*